# AUTOMATED SYSTEM

In [ ]:
!pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface faiss-cpu sentence-transformers

In [ ]:
!pip install networkx leidenalg igraph

In [ ]:
!pip install langchain
!pip install langchain_text_splitters
!pip install openai
!pip install -U "langchain[google-genai]"
!pip install pydantic
!pip install langchain-groq
!pip install neo4j
!pip install networkx
!pip install pyvis
!pip install cdlib
!pip install leidenalg

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from groq import RateLimitError
import json
from google.colab import runtime
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chat_models import init_chat_model
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel,Field
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from neo4j import GraphDatabase
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network
from IPython.display import display, HTML
from cdlib import algorithms
from typing import List, Literal,Union
import time
from tqdm import tqdm # Progress bar
from langchain_core.output_parsers import PydanticOutputParser
import sys
from google.colab import drive
import pandas as pd
import re
from collections import defaultdict
pwd=""
URI = ""
AUTH = ("neo4j", "")
drive.mount('/content/drive')


In [ ]:
def chunked(text):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size=500,
      chunk_overlap=100,
      separators=["\n\n", "\n", " ", ""])
  chunks=splitter.split_text(text)
  return chunks


#### ENTITY RELATION EXTRACTION RULES
EntityType = Literal[
    # CLASS A: BIOLOGICAL AGENTS
    "PATHOGEN",          # Viruses, Bacteria, Fungi (e.g., H3N2, E. coli)
    "HOST",              # Patients, Animals, Demographics (e.g., Refugee patients, Children, Ferrets)

    # CLASS B: MOLECULAR REALITY (Crucial for your data)
    "GENETIC_ENTITY",    # Genes, RNA, DNA, Primers, Amplicons (e.g., HA Gene, PCR Primers)
    "MOLECULAR_VARIANT", # Mutations, Substitutions, Strains, Isolates (e.g., K145N, Nepal Isolates)
    "CHEMICAL",          # Drugs, Proteins, Reagents, Antibodies (e.g., Lisinopril, HA Protein)

    # CLASS C: CLINICAL REALITY
    "CONDITION",         # Diseases, Symptoms, Syndromes (e.g., Influenza, Fever)
    "PHENOMENON",        # Biological processes (e.g., Genetic Drift, Antigenic Variation)

    # CLASS D: INTERVENTION & CONTEXT
    "PROCEDURE",         # Tests, Assays, Sequencing, Vaccinations (e.g., PCR, HI Assay)
    "DEVICE",            # Machines, Kits (e.g., Magnapure LX)
    "LOCATION",          # Geographic or Anatomical (e.g., Nepal, Throat)
    "METRIC",            # Measurements, Titers (e.g., 1:320, 4-fold increase)
    "CELL",
    "ANATOMY",          # Added these 2
    "OTHER"              # Strict fallback
]

RelationType = Literal[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH",    # Fallback for weak correlations
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
]
StudySection = Literal[
    "INTRODUCTION",      # Background, Hypothesis
    "METHODS",           # Study Design, Procedures
    "RESULTS",           # Raw Findings, Data
    "DISCUSSION",        # Interpretation, Limitations
    "CONCLUSION",        # Final Takeaways, Future Work
    "UNKNOWN"            # Fallback
]

class Entity(BaseModel):
    name: str = Field(description="The precise scientific name. For 'Nepal Isolates', extract 'Isolates' as the name and 'Nepal' as a separate Location node.")
    type: EntityType = Field(description="The strict scientific category.If unsure of the category, STRICTLY use 'OTHER'.")

class Relation(BaseModel):
    source_entity: Entity = Field(description="The subject of the relation")
    relation_type: RelationType = Field(description="The strict scientific verb connecting them.If the relationship is complex, you MUST select 'ASSOCIATED_WITH'.")
    target_entity: Entity = Field(description="The object of the relation")
    modality: Literal["CONFIRMED", "HYPOTHETICAL", "NEGATED", "ASSOCIATED"] = Field(
        description="CONFIRMED: Fact. HYPOTHETICAL: 'Suggests/May'. NEGATED: 'Does NOT'. ASSOCIATED: 'Linked to'."
    )
    nuance_comment: str = Field(
        description=
            "A brief, 1-sentence explanation of the relationship's context. "
            "This is where you explain the 'How/Why'."
            "Anything you were not able to mention in relation , DO NOT CREATE INFORMATION"

    )
    study_section: StudySection = Field(
        description="Which part of the scientific paper did this fact come from?"
    )
    evidence: str = Field(description="The EXACT quote from the text supporting this.")
    confidence: float = Field(description="Confidence score (0.0-1.0).")

class MedicalGraph(BaseModel):
   reasoning_trace: str = Field(
        description='''
            "THINKING STEP-BY-STEP LOGIC:\n"
            "Carefully think before answering"
            "DO NOT CREATE INFORMATION , USE INFORMATION PROVIDED IN THE TEXT ONLY."
            ""CONNECTIVITY PLANNING:\n"
            "1. IDENTIFY THE HUB: Which entity is the main subject? (e.g., 'The Patient' or 'The Virus').\n"
            "2. LINK ORPHANS: Look for entities that might become isolated. explicitly plan how to link them back to the Hub.\n"
            '''

    )
   relations: List[Relation]




##### FIRST PROMPT

system_prompt = """You are an expert Medical Knowledge Graph Engineer.
Your goal is to extract a highly precise, scientifically accurate Knowledge Graph from the provided text.

### CORE INSTRUCTION: DECONSTRUCTION OVER LABELING
Medical text often compresses complex relationships into single noun phrases.
Do NOT label complex phrases as single nodes. Instead, DECONSTRUCT them into atomic components.

Examples of Deconstruction:
- "Metastatic Lung Cancer" -> "Lung Cancer" (CONDITION) + "Metastasis" (PHENOMENON).
- "Nepal Isolates" -> "Isolates" (MOLECULAR_VARIANT) + "Nepal" (LOCATION).
- "PCR Primers" -> "Primers" (GENETIC_ENTITY) + "PCR" (PROCEDURE).

### STRICT ENTITY GUIDELINES
1. PATHOGEN: Only the active infectious agent (e.g., "H3N2", "S. aureus"). Do not confuse with the Disease.
2. CONDITION: The passive disease or symptom (e.g., "Influenza", "Pneumonia").
3. CHEMICAL: Includes drugs, proteins, and lab reagents.
4. MOLECULAR_VARIANT: Use this for specific strains, mutations, or receptor statuses (e.g., "Delta Variant", "HER2+").
5. GENETIC_ENTITY: Use this for Genes, RNA, DNA, and Primers.
6. LOCATION: Use this for both Geography (Countries) and Anatomy (Body parts).

**"ASSOCIATED_WITH" TRAP:**
If the relationship is weak or statistical, do NOT use `CAUSES` or `TREATS`. Use `ASSOCIATED_WITH` or `CORRELATES_WITH` and set modality to `ASSOCIATED`.

### RELATIONSHIP USAGE RULES
- Use 'EXHIBITS' to link a Subject to its Properties or Variants (e.g., Virus -> EXHIBITS -> Mutation).
- Use 'TARGETS' for Mechanism of Action (e.g., Drug -> TARGETS -> Protein).
- Use 'ORIGINATES_FROM' to link samples/variants to their source (e.g., Tumor -> ORIGINATES_FROM -> Tissue).
- Use 'HAS_CONTEXT' to link Entities to Metrics, Time, or Locations (e.g., Patient -> HAS_CONTEXT -> Age 60).
- Use 'DETECTS' for Diagnostic tools finding a Condition or Pathogen.

### RELATIONSHIP & MODALITY RULES (THE HALLUCINATION KILLER)
You must select the correct `modality` for every relationship.
- **CONFIRMED:** The text explicitly states a fact. (e.g., "Drug X inhibits Protein Y").
- **HYPOTHETICAL:** The text uses hedging language. (e.g., "Suggests", "May", "Could", "Potential target").
- **NEGATED:** The text explicitly denies a link. (e.g., "No correlation found", "Did not treat").
- **ASSOCIATED:** The text implies a link without mechanism. (e.g., "X is linked to Y", "X co-occurs with Y").

### FIELD USAGE GUIDELINES
- **`nuance_comment`**: This is your "Context Dump".
  - Use it for: Specific conditions ("in mice only"), contradictions ("conflicts with Smith et al"), or statistical details ("p < 0.05").
  - Do NOT create new info, but preserve the richness here.
- **`study_section`**: Infer the section based on tone.
  - "We hypothesize..." -> INTRODUCTION.
  - "We measured..." -> METHODS.
  - "Data showed..." -> RESULTS.
  - "This suggests..." -> DISCUSSION/CONCLUSION.

### CONNECTIVITY PLANNING (`reasoning_trace`)
Before generating relations, use the `reasoning_trace` field to:
1.  **Identify the Hub:** What is the central topic?
2.  **Plan Connectivity:** Avoid orphan nodes.Ensure every extracted node links back to the hub or a main chain.

### OUTPUT LOGIC
- If the text mentions "Resistance to Tamiflu", extract:
  (Tamiflu: CHEMICAL) -[TREATS]-> (Influenza: CONDITION)
  (Influenza: CONDITION) -[EXHIBITS]-> (Resistance: PHENOMENON)
  (Resistance: PHENOMENON) -[ASSOCIATED_WITH]-> (Tamiflu: CHEMICAL)

Analyze the text deeply. Bias towards granular, scientifically precise nodes.
"""

few_shot_examples = """
Input: "Our study suggests that H3N2 mutations in Nepal isolates might cause resistance to Oseltamivir."
Output: {
  "reasoning_trace": "Hub: H3N2. Deconstruction: 'Nepal isolates' -> 'Isolates' + 'Nepal'. Modality Check: 'Suggests/Might' -> HYPOTHETICAL.",
  "relations": [
    {
      "source_entity": {"name": "Isolates", "type": "MOLECULAR_VARIANT"},
      "relation_type": "ORIGINATES_FROM",
      "target_entity": {"name": "Nepal", "type": "LOCATION"},
      "modality": "CONFIRMED",
      "nuance_comment": "Specific strain origin",
      "study_section": "RESULTS",
      "evidence": "Nepal isolates",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "H3N2", "type": "PATHOGEN"},
      "relation_type": "EXHIBITS",
      "target_entity": {"name": "Mutation", "type": "MOLECULAR_VARIANT"},
      "modality": "CONFIRMED",
      "nuance_comment": "",
      "study_section": "RESULTS",
      "evidence": "H3N2 mutations",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "Mutation", "type": "MOLECULAR_VARIANT"},
      "relation_type": "CAUSES",
      "target_entity": {"name": "Resistance", "type": "PHENOMENON"},
      "modality": "HYPOTHETICAL",
      "nuance_comment": "Text says 'might cause', indicating uncertainty.",
      "study_section": "CONCLUSION",
      "evidence": "mutations... might cause resistance",
      "confidence": 0.7
    },
    {
      "source_entity": {"name": "Resistance", "type": "PHENOMENON"},
      "relation_type": "ASSOCIATED_WITH",
      "target_entity": {"name": "Oseltamivir", "type": "CHEMICAL"},
      "modality": "CONFIRMED",
      "nuance_comment": "Context of resistance",
      "study_section": "CONCLUSION",
      "evidence": "resistance to Oseltamivir",
      "confidence": 0.9
    }
  ]
}

Input: "Patient denies chest pain but reports severe shortness of breath."
Output: {
  "reasoning_trace": "Hub: Patient. Negation Check: 'denies chest pain' -> Modality NEGATED. Attribute: 'severe' -> nuance_comment.",
  "relations": [
    {
      "source_entity": {"name": "Patient", "type": "HOST"},
      "relation_type": "EXHIBITS",
      "target_entity": {"name": "Chest Pain", "type": "SYMPTOM"},
      "modality": "NEGATED",
      "nuance_comment": "Patient explicitly denied this symptom.",
      "study_section": "RESULTS",
      "evidence": "denies chest pain",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "Patient", "type": "HOST"},
      "relation_type": "EXHIBITS",
      "target_entity": {"name": "Shortness of Breath", "type": "SYMPTOM"},
      "modality": "CONFIRMED",
      "nuance_comment": "Described as severe.",
      "study_section": "RESULTS",
      "evidence": "reports severe shortness of breath",
      "confidence": 1.0
    }
  ]
}
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Here are examples of the expected JSON output format:\n{examples}"),
    ("human", "Analyze this clinical text and extract the graph:\n{text}")
])


####ADDING NEO4J

merge_query="""
MERGE (e:Entity{name:$name1})
ON CREATE
  SET e.type=$type1

MERGE (f:Entity{name:$name2})
ON CREATE
  SET f.type=$type2

MERGE (e)-[r:RELATIONSHIP]->(f)
ON CREATE
  SET r.modality=$modality,
      r.Nuance=$nuance,
      r.study_section=$study,
      r.evidence=$evidence
"""
relationships=[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH",    # Fallback for weak correlations
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
]

def add_neo4j(final_results,merge_query,URI,AUTH):
  x=json.dumps(final_results)
  x=json.loads(x)
  with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    #record,summary,key=driver.execute_query("MATCH (p:Person) RETURN p.name",database_="neo4j")  ---------------->>>> for read operation
    record,summary,key=driver.execute_query("MATCH(n) DETACH DELETE n",database_="neo4j")
    for i in x:
      for j in i['relations']:
        source_name=j['source_entity'].get('name')
        source_type=j['source_entity'].get('type')
        target_name=j['target_entity'].get('name')
        target_type=j['target_entity'].get('type')
        relationship=j['relation_type']
        modality=j['modality']
        nuance=j['nuance_comment']
        study_section=j['study_section']
        evidence=j['evidence']
        if not (source_name and target_name and relationship):
                continue
        if relationship in relationships:
          final_query=merge_query.replace("RELATIONSHIP",relationship)
          summary=driver.execute_query(final_query,name1=source_name,type1=source_type,name2=target_name,type2=target_type,modality=modality,nuance=nuance,study=study_section,evidence=evidence,database_="neo4j")
        else:
          print("----LLM HALLUCINATED AND NOT FOLLOWED ORDERS-----------------------------------------------------")


### EXTRACTING BACK FROM NEO4J
def communties_extracton(g,URI,AUTH):
  with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    records,summary,key=driver.execute_query("MATCH (n:Entity)-[r]->(m:Entity) RETURN n,r,m,r.modality,r.Nuance,r.study_section")
  for record in records:
    x=record.data()
    source=x.get("n")
    source_nxname=str(source["name"])
    source_nxtype=source["type"]
    rel=x.get("r")
    rel=rel[1]
    target=x.get("m")
    target_nxname=str(target["name"])
    target_nxtype=target["type"]
    rel_modal=x.get('r.modality')
    rel_nuance=x.get('r.Nuance')
    rel_study=x.get('r.study_section')
    g.add_node(source_nxname)
    g.add_node(target_nxname)
    g.add_edge(source_nxname,target_nxname)
    g.nodes[source_nxname]["type"]=source_nxtype
    g.nodes[target_nxname]["type"]=target_nxtype
    g.edges[source_nxname,target_nxname]["name"]=rel
    g.edges[source_nxname,target_nxname]["Modality"]=rel_modal
    g.edges[source_nxname,target_nxname]["Nuance"]=rel_nuance
    g.edges[source_nxname,target_nxname]["Study_Section"]=rel_study
  print("done")
  communities = algorithms.leiden(g)
  return communities


system_template = """You are a specialized Medical Science Communicator.
Your task is to turn raw graph data into a professional, distinct, and grammatically fluent clinical summary.

### 1. THE CONSTRAINT (CLOSED WORLD)
- You may ONLY use the entities and relationships provided in the input.
- Do NOT bring in outside medical facts (no hallucinations). DO NOT USE PRETRAINED MEDICAL KNOWLEDGE
- **VERBATIM ADHERENCE:** You must respect the *strength* of the relationship provided.

### 2. THE "ASSOCIATED_WITH" TRAP (CRITICAL)
- If the relationship is `ASSOCIATED_WITH`, `HAS_CONTEXT`, or `RELATED_TO`:
  - **DO NOT** imply causation (e.g., do NOT write "leads to", "causes", "triggers").
  - **DO NOT** invent a mechanism (e.g., do NOT write "via the activation of...").
  - **MUST USE** neutral language: "is linked to", "co-occurs with", "is present in the context of".
  - *Example:* If Input is `(Drug A) -> [ASSOCIATED_WITH] -> (Symptom B)`, it means "Drug A is associated with Symptom B." and nothing else (Do NOT write "Drug A causes Symptom B").

### 3. THE FLUENCY RULES (CRITICAL FOR GOOD GRAMMAR)
- **No Robotic Lists:** Do not write "A causes B. B causes C."
- **Use Compound Sentences:** Combine related edges into fluid thoughts.
  - *Bad:* "Obesity causes Diabetes. Diabetes causes Renal Failure."
  - *Good:* "Obesity is a driver of Diabetes , which subsequently leads to Renal Failure ."
- **Vary Sentence Structure:** Use transition words (e.g., "Furthermore," "Consequently," "In contrast").

### HOW TO DECODE THE INPUT (CRITICAL)
The input format is: `(Node A) -[RELATION]-> (Node B) <MODALITY: M | STUDY_SECTION: S | NUANCE: N>`

**A. MODALITY RULES (The Truth Filter):**
- **CONFIRMED:** State as fact. ("A causes B").
- **HYPOTHETICAL:** Use hedging. ("A *may* cause B", "Evidence suggests A links to B").
- **NEGATED:** Explicitly state the lack of link. ("A does *not* cause B", "A is ruled out").
- **ASSOCIATED:** Use neutral language. ("A is observed alongside B").

**B. THE "NUANCE" FIELD (The Context):**
- You MUST incorporate the `Note` content into the sentence to add depth.
- *Example:* Input: `(Drug) -> [TREATS] -> (Disease) <Note: in mice only>`
- *Output:* "The drug treats the disease *in murine models*."

**C. THE "STUDY_SECTION" FIELD (The Narrative Frame):**
- Use the `Section` to frame *when* the information is presented.
- *INTRODUCTION:* "Background data indicates..."
- *RESULTS:* "Clinical findings show..."
- *CONCLUSION:* "The study concludes that..."


Here is the strict graph data you must convert into text.

### PART A: KNOWN ENTITIES (node_list)
{node}

### PART B: VERIFIED EDGES (relationship_list)
{relationship}

### YOUR TASK:
Perform this task in two distinct steps using XML tags:

STEP 1: <thinking>
1. **Cluster Analysis:** Group the edges into logical themes (e.g., "Pathology Progression", "Treatment Regimen", "Symptoms").
2. **Chain Building:** Identify long paths (A -> B -> C) that can be merged into a single fluid sentence.
3. **Orphan Check:** Identify any isolated nodes and plan how to weave them into the context naturally.
4.""INCLUSION:** Try to include every node into the summary
5. **Fact Check:** Confirm that every planned sentence is supported by a specific edge in Part B.

STEP 2: <summary>
Write the precise, grounded clinical summary based on your plan. Retain the exact node name DO NOT CHANGE IT. DO NOT CREATE INFORMATON OR USE PRETRAINED KNOWLEDGE. Try to use all the NUMERIC metrics provided as it is DO NOT CHANGE THE METRICS.
Do not mention the graph structure (e.g., "Node A connects to Node B")—write as a doctor speaking to a doctor.
</summary>
"""

strict_graph_prompt = PromptTemplate(
    template=system_template,
    input_variables=["node","relationship"]
)



def five_seasons(g,strict_graph_prompt,communities):
  community_summary=[]
  pattern = r"<summary>(.*?)</summary>"
  llm = ChatGroq(
      model="llama-3.3-70b-versatile",
      temperature=0,
      max_retries=2,
  )
  chain2= strict_graph_prompt | llm
  for i,member in enumerate(communities.communities):
    community_relations=[]
    community_nodes=[]
    subgraph=g.subgraph(member)
    community_node=member
    for j in subgraph.edges:
      rel=subgraph.edges[j]["name"]
      rel_modal=subgraph.edges[j]["Modality"]
      rel_nuance=subgraph.edges[j]["Nuance"]
      rel_study=subgraph.edges[j]["Study_Section"]
      relation=j[0]+" "+rel+" "+j[1]+"METADATA FOR RELATION< MODALITY: "+rel_modal+" | STUDY_SECTION: "+ rel_study+" | NUANCE: "+rel_nuance+">"
      if relation not in community_relations:
        community_relations.append(relation)
    community_response=chain2.invoke({"node":community_node,"relationship":community_relations})
    match = re.search(pattern, community_response.content, re.DOTALL)
    if match:
      community_summary.append(match.group(1).strip())
    else:
      community_summary.append(community_response.content)
    if i>4:
      break
  return community_summary






def get_injection_nodes(G, partition, top_n=5, inject_count=10):
    community_map = defaultdict(list)
    for node, comm_id in partition.items():
        community_map[comm_id].append(node)
    sorted_communities = sorted(community_map.items(), key=lambda x: len(x[1]), reverse=True)
    top_community_ids = {comm_id for comm_id, nodes in sorted_communities[:top_n]}
    tail_nodes = []
    for comm_id, nodes in sorted_communities[top_n:]:
        tail_nodes.extend(nodes)

    if not tail_nodes:
        return []
    pagerank_scores = nx.pagerank(G)
    tail_scores = {node: pagerank_scores[node] for node in tail_nodes}
    sorted_tail_nodes = sorted(tail_scores.items(), key=lambda x: x[1], reverse=True)
    injection_list = [node for node, score in sorted_tail_nodes[:inject_count]]
    return injection_list


INJECTION_PROMPT = PromptTemplate(template="""
You are a Medical Data Analyst.
You have been provided with a list of "orphan" medical entities extracted from the tail end of a patient's knowledge graph. These entities are disconnected from the main clinical narrative but are crucial for completeness.

### INPUT DATA (Raw Entity List)
{raw_node_list}

### YOUR TASK
Convert this raw list into a structured "Additional Clinical Findings" section.

### GUIDELINES
1. **Filter Noise:** Remove any non-medical terms (e.g., "Date", "Page 2", "Hospital Name", generic words like "Patient"). Keep ONLY distinct clinical concepts.
2. **Categorize:** Group the entities logically (e.g., "Medications," "Comorbidities," "Lab Values," "Symptoms").
3. **Neutral Phrasing:** Do NOT invent relationships.
   - *Bad:* "The patient took Aspirin because of the Headache." (Hallucination risk)
   - *Good:* "The dataset also notes the presence of Aspirin and Headache."
4. **Keyword Preservation:** Ensure the exact names of the important entities appear in the output to maintain semantic fidelity.

### OUTPUT FORMAT
Provide the output in a single block using the following structure:

**Additional Clinical Notations**
Analysis of the remaining graph data highlights several isolated clinical concepts.
* **[Category 1]:** [Entity A], [Entity B]
* **[Category 2]:** [Entity C], [Entity D]
* **Other Findings:** [Entity E]
""",input_variables=["raw_node_list"])




final_summary_template = PromptTemplate(
    template='''
You are a Medical Archivist and Clinical Compiler.
You have received several partial reports and a list of isolated entities.
Your goal is to weave EVERY single fact from these inputs into one master document.

### 1. THE "ZERO COMPRESSION" MANDATE
- **DO NOT SUMMARIZE.** Your output should be roughly the same length as the combined inputs.
- **DO NOT DELETE DETAILS.** If Input A mentions "left ventricular hypertrophy" and Input B mentions "LVEF 45%", the final report MUST contain both.
- **REDUNDANCY CHECK:** Only remove information if it is an *exact* word-for-word repetition. If it adds even a tiny nuance, KEEP IT.

### 2. THE "DIRECT REUSE" RULE
- You are strictly required to **COPY AND PASTE** complete sentences from the input if they are grammatically sound.
- Do not rewrite a perfect medical sentence just to sound different. Preserve the original phrasing to ensure accuracy.

### 3. INPUT DATA
"""
{community_summaries_text}
"""
### 4. REDUNDANCY CHECK
IF multiple inputs state the exact same fact (e.g., "Dose was 10mg"), do NOT simply delete the duplicates.INSTEAD, merge them into a single confirmed statement
IF Input A provides a general claim (e.g., "BP improved") and Input B provides a specific metric (e.g., "BP dropped by 10mmHg"), you MUST preserve the specific metric.
Never delete a sentence if it contains a number, p-value, sample size, or dosage that appears nowhere else in the text.
Even if the surrounding text seems redundant, the number itself is a unique data point that must be preserved.
NEVER EVER CREATE YOUR OWN INFORMATION , ALWAYS USE THE INPUT DATA

### 5. THE COMPILATION TASK (Chain of Thought)
Perform this in two steps using XML tags(MAKE SURE YOU USE THEM):

STEP 1: <thinking>
1. **Scope Audit:** Scan the inputs. Acknowledge that you must preserve ALL specific metrics (dates, dosages, counts).
2. **Structure Design:** organizing the "Narrative" inputs into a logical flow (e.g., Clinical Presentation -> Diagnostics -> Therapeutic Interventions).
3. **Integration of Isolated Nodes:** Look at the "isolated/tail" list (the non-sentence part of the input). Can any of these items be naturally inserted into the narrative sections?
   - *Yes:* Plan to insert them where they fit.
   - *No:* Plan to list them in the "Additional Findings" section.
4. **Safety Check:** Confirm you are not adding any outside information.

STEP 2: <output>
Write the Master Clinical Report.(DO NOT ADD ANYTHING IRRELEVANT)(**RETAIN ALL THE NUMERIC METRICS**)
- **Section 1: Comprehensive Clinical Narrative** (A dense, detailed account. Use transition words to stitch the input sentences together, but do not shorten them).
- **Section 2: Additional Clinical Notations** (List any remaining entities from the isolated list that could not be woven into Section 1).
</output>

BEGIN:
    ''',
    input_variables=["community_summaries_text"]
)


In [ ]:
### pydantic fail logic
import json
import re

def extract_relations_from_error(e):
    failed_gen = None

    # 1. Extraction Strategy (Same as your code)
    if hasattr(e, "error") and isinstance(e.error, dict):
        failed_gen = e.error.get("failed_generation")
    if failed_gen is None and hasattr(e, "body"):
        body = e.body
        if isinstance(body, dict):
            failed_gen = body.get("error", {}).get("failed_generation")
    if failed_gen is None and e.args:
        arg0 = e.args[0]
        if isinstance(arg0, dict):
            failed_gen = arg0.get("error", {}).get("failed_generation")

    if failed_gen is None:
        # Fallback regex search
        text = str(e)
        match = re.search(r"<function=MedicalGraph>(.*?)</function>", text, re.DOTALL)
        if match:
            failed_gen = match.group(1)

    if failed_gen is None:
        raise ValueError("failed_generation not found anywhere")

    start = failed_gen.find("{")
    end = failed_gen.rfind("}") + 1
    json_str = failed_gen[start:end]
    try:
        data = json.loads(json_str, strict=False)
    except json.JSONDecodeError:
        try:
            sanitized_str = json_str.replace('\n', ' ').replace('\r', '')
            data = json.loads(sanitized_str, strict=False)
        except Exception as final_e:
            print(f"CRITICAL JSON FAILURE. RAW STRING:\n{json_str}")
            raise final_e


    return [
        {
            "source_entity": r["source_entity"]["name"],
            "source_type": r["source_entity"].get("type", "OTHER"),
            "relation": r["relation_type"],
            "target_entity": r["target_entity"]["name"],
            "target_type": r["target_entity"].get("type", "OTHER"),
            "evidence": r.get("evidence", ""),
            "confidence": r.get("confidence"),
            "modality":r.get("modality"),
            "nuance_comment":r.get("nuance_comment"),
            "study_section":r.get("study_section")

        }
        for r in data.get("relations", [])
    ]
VALID_RELATIONS = {
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITON
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION
    "EXHIBITS",          # PATHOGEN -> VARIANT
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENTIC_ENTITY
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN
    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST
    "DIFFERS_FROM",      # VARIANT -> VARIANT
    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME
    "ASSOCIATED_WITH",   # Fallback (weak correlation)
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
}
def normalize_relation(relation: str) -> str:
    if relation in VALID_RELATIONS:
        return relation
    return "ASSOCIATED_WITH"
def normalize_relations(relations):
    normalized = []
    for r in relations:
        original = r["relation"]
        fixed = normalize_relation(original)
        if original != fixed:
            print(f"[RELATION FIX] {original} → {fixed}") ###delete this if want to
        r["relation"] = fixed
        normalized.append(r)
    return normalized
def convert_relations_to_canonical(relations):
    canonical_block = {"relations": []}
    for r in relations:
        canonical_block["relations"].append({
            "source_entity": {
                "name": r["source_entity"],
                "type": r.get("source_type", "OTHER")
            },
            "relation_type": r["relation"],
            "target_entity": {
                "name": r["target_entity"],
                "type": r.get("target_type", "OTHER")
            },
            "evidence": r.get("evidence", ""),
            "confidence": float(r.get("confidence", 0.7)),
            "modality":r.get("modality"),
            "nuance_comment":r.get("nuance_comment"),
            "study_section":r.get("study_section")
        })
    return canonical_block

In [ ]:
class EntityNumber(BaseModel):
    name: Union[int, float,str] = Field(
        description="The EXACT numerical value found in the text (e.g., 10, 0.05, 320). Do not round or summarize.If this is a 'Target Anchor', use the scientific term as a string."
    )
    type: EntityType = Field(
        description="The strict scientific category this number represents. If it is a measurement, use 'METRIC'."
    )

class RelationNumber(BaseModel):
    source_entity: EntityNumber = Field(
        description="The subject of the relation (usually the entity being measured or the baseline value)."
    )
    relation_type: RelationType = Field(
        description="The verb connecting the number to the target. Use 'MEASURES' or 'HAS_CONTEXT' for simple data points."
    )
    target_entity: EntityNumber = Field(
        description="The semantic anchor (e.g., 'Efficacy Rate') or the main topic node."
    )
    modality: Literal["CONFIRMED", "HYPOTHETICAL", "NEGATED", "ASSOCIATED"] = Field(
        description="CONFIRMED: Fact. HYPOTHETICAL: 'Suggests/May'. NEGATED: 'Does NOT'. ASSOCIATED: 'Linked to'."
    )
    nuance_comment: str = Field(
        description="MANDATORY: Link the number to its physical context (e.g., '120 is the systolic BP at rest'). Do not create info."
    )
    study_section: StudySection = Field(
        description="Which part of the scientific paper did this fact come from?"
    )
    evidence: str = Field(
        description="The EXACT quote containing the number and its immediate context."
    )
    confidence: float = Field(description="Confidence score (0.0-1.0).")

class MedicalGraphNumber(BaseModel):
    reasoning_trace: str = Field(
        description='''
            "THINKING STEP-BY-STEP LOGIC:\n"
            "1. SCAN: List every digit in the text.\n"
            "2. FILTER: Ignore citation years (e.g., 2023, 2024), Table numbers, and Page numbers.\n"
            "3. ANCHORING: For every number, identify the 'Anchor' (what it measures).\n"
            "4. NULL CHECK: If no clinical numbers found, return relations: []."
            '''
    )
    relations: List[RelationNumber]= Field(description="List of relations. Leave EMPTY if no numerical data is found.")


system_prompt_numerical = """You are an expert Medical Knowledge Graph Engineer. Your mission is to extract numerical values with forensic precision, ensuring they are anchored to clinical subjects to prevent graph fragmentation.

### THE CITATION & METADATA FILTER
- **STRICT EXCLUSION:** You MUST ignore all citation years (e.g., "Smith (2023)", "Zhao et al., 2024").
- **STRICT EXCLUSION:** Ignore Table, Figure, and Page references (e.g., "Table 1", "Fig. 2").
- **FOCUS:** Only extract numbers representing clinical data: p-values, dosages, patient counts, concentrations, ratios, and titers.

### THE ANCHORING RULE (Anti-Fragmentation)
To prevent "floating" numbers that the GraphRAG system cannot find, you must create a "Hub-and-Spoke" chain:
1. Every Number (int/float) must be a Source Entity.
2. Every Number MUST be linked to a "Target Anchor" (the scientific name of the metric, e.g., "Survival Rate").
3. That "Target Anchor" MUST then be linked to the "Main Subject" of the chunk (e.g., "H3N2" or "Metformin").

### CORE RULE: ZERO SMOOTHING
- Never replace a digit with a word (e.g., do not change "98%" to "almost all").
- If the text says "0.045", you must extract 0.045.
- If NO clinical numbers exist, return: {"reasoning_trace": "NO_NUMERICAL_DATA_FOUND", "relations": []}.

### RELATIONSHIP & MODALITY
- Use 'MEASURES' to link a Number to its Metric Anchor.
- Use 'HAS_CONTEXT' or 'ASSOCIATED_WITH' to link the Anchor to the Pathogen/Condition.
- **CONFIRMED:** Explicit facts.
- **HYPOTHETICAL:** "Suggests", "May", "Expected".

### CONNECTIVITY PLANNING (reasoning_trace)
1. **FILTER:** List all numbers found, then explicitly state which are citations (to be ignored) and which are clinical data.
2. **ANCHORING:** Map each clinical number to a specific metric name.
3. **HUB:** Identify the central clinical subject to link everything together.
"""

few_shot_examples_numerical = """
Input: "According to Zhao et al. (2024), the H3N2 viral load dropped significantly by 4.2 logs (p < 0.005) within 72 hours."
Output: {
  "reasoning_trace": "1. SCAN: 2024, 4.2, 0.005, 72. 2. FILTER: '2024' is a citation year (IGNORE). 3. HUB: H3N2. 4. ANCHORING: Link 4.2 to 'Viral Load Reduction', 0.005 to 'Statistical Significance', and 72 to 'Time Duration'.",
  "relations": [
    {
      "source_entity": {"name": 4.2, "type": "METRIC"},
      "relation_type": "MEASURES",
      "target_entity": {"name": "Viral Load Reduction", "type": "PHENOMENON"},
      "modality": "CONFIRMED",
      "nuance_comment": "Log base 10 reduction in H3N2 particles.",
      "study_section": "RESULTS",
      "evidence": "viral load dropped significantly by 4.2 logs",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "Viral Load Reduction", "type": "PHENOMENON"},
      "relation_type": "HAS_CONTEXT",
      "target_entity": {"name": "H3N2", "type": "PATHOGEN"},
      "modality": "CONFIRMED",
      "nuance_comment": "Metric specifically measures H3N2 decline.",
      "study_section": "RESULTS",
      "evidence": "H3N2 viral load dropped",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": 0.005, "type": "METRIC"},
      "relation_type": "MEASURES",
      "target_entity": {"name": "P-Value", "type": "METRIC"},
      "modality": "CONFIRMED",
      "nuance_comment": "Statistical significance of the viral load drop.",
      "study_section": "RESULTS",
      "evidence": "p < 0.005",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": 72, "type": "METRIC"},
      "relation_type": "MEASURES",
      "target_entity": {"name": "Treatment Duration", "type": "METRIC"},
      "modality": "CONFIRMED",
      "nuance_comment": "Hours elapsed before measurement.",
      "study_section": "METHODS",
      "evidence": "within 72 hours",
      "confidence": 1.0
    }
  ]
}

Input: "We observed a 15% increase in resistance markers in the 2023 Nepal isolates compared to baseline."
Output: {
  "reasoning_trace": "1. SCAN: 15, 2023. 2. FILTER: '2023' is a temporal isolate identifier/year (IGNORE to prevent citation hubs). 3. HUB: Nepal Isolates. 4. ANCHORING: Link 15 to 'Resistance Increase'.",
  "relations": [
    {
      "source_entity": {"name": 15, "type": "METRIC"},
      "relation_type": "MEASURES",
      "target_entity": {"name": "Resistance Marker Increase", "type": "PHENOMENON"},
      "modality": "CONFIRMED",
      "nuance_comment": "Percentage increase in markers compared to baseline.",
      "study_section": "RESULTS",
      "evidence": "15% increase in resistance markers",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "Resistance Marker Increase", "type": "PHENOMENON"},
      "relation_type": "ASSOCIATED_WITH",
      "target_entity": {"name": "Isolates", "type": "MOLECULAR_VARIANT"},
      "modality": "CONFIRMED",
      "nuance_comment": "Relates to Nepal variant isolates.",
      "study_section": "RESULTS",
      "evidence": "resistance markers in the... Nepal isolates",
      "confidence": 0.95
    }
  ]
}

Input: "Previous literature reviews discuss the general mechanism of viral entry without providing new titration data."
Output: {
  "reasoning_trace": "NO_NUMERICAL_DATA_FOUND: No clinical digits or quantitative metrics found. Purely descriptive text.",
  "relations": []
}
"""

prompt_template_numerical = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Here are some examples of correct extraction: {examples}"),
    ("human", "Analyze this clinical note: {text}")
])

In [ ]:
import re

def has_relevant_numbers(text: str) -> bool:
    """
    Returns True if the text contains:
    - Decimals (0.05)
    - Percentages (50%)
    - Math/Ranges (10-20, 1/2, 5*, +3)
    - Integers that are likely NOT years (matches 5, 100; ignores 2024, 1999)
    """
    pattern = r"""
        \d+\.\d+                # Matches Decimals (0.05, 12.5)
        |                       # OR
        %                       # Matches Percent signs
        |                       # OR
        \d+\s*[-+*/]\s*\d+      # Matches Ranges/Fractions/Math (10-20, 10/2, 5*5, 1+1)
        |                       # OR
        [-+]\d+                 # Matches Signed Numbers (-5, +10)
        |                       # OR
        \d+[-+*/]               # Matches Trailing Signs (10+, 5*, HER2+)
        |                       # OR
        :\d+                    # Matches Ratios (1:640)
        |                       # OR
        \b(?!(?:19|20)\d{2}\b)\d+\b  # Matches Integers, EXCLUDING Years (19xx, 20xx)
    """
    return bool(re.search(pattern, text, re.VERBOSE))

In [ ]:
#naive rag
import os
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

def get_naive_rag_summary(article_text,model_name="llama-3.3-70b-versatile"):
    llm = ChatGroq(
        temperature=0,
        model_name=model_name
    )
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )
    docs = splitter.create_documents([article_text])

    vectorstore = FAISS.from_documents(
        docs,
        embeddings
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={"k":3}
    )
    query = "Summarize this medical article"
    retrieved_docs = retriever.invoke(query)
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )
    prompt = ChatPromptTemplate.from_messages([

        ("system",
"""
You are a medical research assistant.

Create a concise structured summary.

Focus on:

1 Objective of study
2 Key numeric findings
3 Complications
4 Certainty of evidence

Use ONLY provided context.

Context:
{context}
"""
),

("human",
"Summarize the article.")
    ])

    formatted_prompt = prompt.format_messages(
        context=context
    )
    response = llm.invoke(formatted_prompt)
    return response.content


# try:
#   article=""
#   summary = get_naive_rag_summary(article)

#   print("\nSUMMARY:\n")
#   print(summary)

# except Exception as e:
#   print("Error :", e)

In [ ]:
##naive graph rag
import os
import json
import igraph as ig
import leidenalg
import networkx as nx
from groq import Groq
from sentence_transformers import SentenceTransformer
from groq import RateLimitError


embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)



def chunk_text(article, chunk_size=900, overlap=100):
    chunks = []
    start = 0
    while start < len(article):
        end = start + chunk_size
        chunks.append(article[start:end])
        start = end - overlap
    return chunks


def llm_call(prompt):

    llm= ChatGroq(

        model="llama-3.3-70b-versatile",
        temperature=0

    )
    response=llm.invoke(prompt).content

    return response


def extract_entities_relations(chunk):

    prompt=f"""
Extract entities and relations from text.

Return STRICT JSON:

{{
"entities":["entity1","entity2"],

"relations":[
{{"source":"A",
"target":"B",
"relation":"causes"}}
]
}}

TEXT:
{chunk}
"""

    output = llm_call(prompt)
    try:
        data=json.loads(output)
    except:
        return [],[]
    return data.get("entities",[]),data.get("relations",[])

def build_graph(chunks):
    G = nx.Graph()
    for chunk in chunks:
        entities, relations = extract_entities_relations(chunk)
        for e in entities:
            G.add_node(e)
        for r in relations:
            G.add_edge(
                r["source"],
                r["target"],
                relation=r["relation"]
            )
    return G
def leiden_partition(graph):
    if len(graph.nodes)==0:
        return []
    mapping = {
        node:i for i,node in enumerate(graph.nodes)
    }
    edges = [
        (mapping[u],mapping[v])
        for u,v in graph.edges
    ]
    ig_graph = ig.Graph(
        edges=edges,
        directed=False
    )
    partition = leidenalg.find_partition(
        ig_graph,
        leidenalg.ModularityVertexPartition
    )
    reverse_map = {
        v:k for k,v in mapping.items()
    }
    communities=[]
    for comm in partition:
        communities.append(
            [reverse_map[i] for i in comm]
        )
    return communities

def summarize_community(nodes):
    text=" , ".join(nodes)
    prompt=f"""

Summarize the medical meaning of these related entities:
{text}
Explain objective, outcomes and risks.

"""
    return llm_call(prompt)

def final_summary(community_summaries):
    combined="\n\n".join(community_summaries)
    prompt=f"""
Create final medical research summary from:
{combined}
Include:

Objective
Findings
Complications
Evidence certainty.

"""
    return llm_call(prompt)

def naive_graph_rag_summary(article):
    chunks = chunk_text(article)
    graph = build_graph(chunks)
    communities = leiden_partition(graph)
    summaries=[]
    for community in communities:
        summaries.append(
            summarize_community(community)
        )
    return final_summary(summaries)

# try:
#   summary = get_naive_rag_summary(article)
#   print(summary)
# except Exception as e:
#   print("")



In [ ]:
current_abstract=68 ### change this accordingly
from pydantic import ValidationError
model1="llama-3.3-70b-versatile"
# model="openai/gpt-oss-120b"
model="llama-3.3-70b-versatile"

folder_path = '/content/drive/MyDrive/Graph_Rag_Testing'
file_name='BENCHMARKING.xlsx'
if os.path.exists(folder_path):
  file_path=os.path.join(folder_path,file_name)
  df=pd.read_excel(file_path)

while current_abstract < len(df):
  article=df.iloc[current_abstract,1]
  chunks=chunked(article)
  llm = ChatGroq(model=model1,temperature=0,max_retries=2,)

  structured_llm = llm.with_structured_output(MedicalGraph)

  chain = prompt_template | structured_llm

  structured_llm_numerical=llm.with_structured_output(MedicalGraphNumber)
  chain_numerical=prompt_template_numerical | structured_llm_numerical

  final_results = []
  errors = []

  print(f"Starting extraction on {len(chunks)} chunks...")

  for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
    while True:
        try:
            result = chain.invoke({
                "examples": few_shot_examples,
                "text": chunk
            })

            final_results.append(result.model_dump())
            break

        except RateLimitError:
          if current_key<len(groq_key)-1:
            current_key+=1
            print("KEY CHANGED-----------------------------------------------,current key:",current_key)
          else:
            runtime.unassign()
            time.sleep(10)
            sys.exit()
          os.environ["GROQ_API_KEY"]=groq_key[current_key]
          llm = ChatGroq(model=model1,temperature=0,max_retries=2,)
          structured_llm = llm.with_structured_output(MedicalGraph)
          chain = prompt_template | structured_llm

        except Exception as e:
          print(f"Error on chunk {i}: {e}")
          errors.append({"chunk_index": i, "error": str(e)})
          try:
            relations = extract_relations_from_error(e)
            relations = normalize_relations(relations)
            canonical = convert_relations_to_canonical(relations)
            final_results.append(canonical)
            print("Successful")
            break
          except Exception as e3:
            print("UnSuccessful")
            break
    while True:
        try:
          if not has_relevant_numbers(chunk):
            print("SAVED API COST")
            break
          print("EXTRACTING NUMBER IF ANY")
          result1 = chain_numerical.invoke({"examples": few_shot_examples_numerical,"text": chunk})
          final_results.append(result1.model_dump())
          break
        except RateLimitError:
          if current_key<len(groq_key)-1:
            current_key+=1
            print("KEY CHANGED-----------------------------------------------,current key:",current_key)
          else:
            runtime.unassign()
            time.sleep(10)
            sys.exit()
          os.environ["GROQ_API_KEY"]=groq_key[current_key]
          llm = ChatGroq(model=model1,temperature=0,max_retries=2,)
          structured_llm_numerical=llm.with_structured_output(MedicalGraphNumber)
          chain_numerical=prompt_template_numerical | structured_llm_numerical

        except Exception as e:
          print(f"Error on chunk {i}: {e}")
          errors.append({"chunk_index": i, "error": str(e)})
          break
    time.sleep(4)
  add_neo4j(final_results,merge_query,URI,AUTH)
  g=nx.Graph()
  communities=communties_extracton(g,URI,AUTH)
  while True:
    try:
      community_summary=five_seasons(g,strict_graph_prompt,communities)
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED-----------------------------------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  while True:
    try:
      communities_injection = algorithms.leiden(g)
      partition = {}
      for i, community_list in enumerate(communities_injection.communities):
        for node in community_list:
          partition[node] = i
      injected_nodes = get_injection_nodes(g, partition, top_n=5, inject_count=15)
      llm = ChatGroq(model=model,temperature=0,max_retries=2,)
      chain_inject= INJECTION_PROMPT | llm
      injection_response=chain_inject.invoke({"raw_node_list":injected_nodes})
      community_summary.append(injection_response.content)

      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED-----------------------------------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  while True:
    try:
      llm = ChatGroq(model=model,temperature=0,max_retries=2,)
      chain3= final_summary_template | llm
      final_response=chain3.invoke({"community_summaries_text":community_summary})
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED------------------------&&&&&&^^^^^^^%%%%%%%%%%%-----------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  pattern = r"<output>(.*?)</output>"
  match = re.search(pattern, final_response.content, re.DOTALL)
  if match:
    print(match.group(1).strip())
    df.at[current_abstract, 'SYSTEM']=match.group(1).strip()
  else:
    print(final_response.content)
    df.at[current_abstract, 'SYSTEM']=final_response.content

  print("UPDATED SUCESSFULLY------------------------$$$$$$$$$$$$$$$%%%%%%%%%%%%%%%%%%%%-----------------------------")
  while True:
    try:
      summary = naive_graph_rag_summary(article)
      print("NAIVE GRAPH RAG DONE_____________________________________________________________")
      df.at[current_abstract, 'naïve_graphrag']=summary
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED------------------------&&&&&&^^^^^^^%%%%%%%%%%%-----------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  while True:
    try:
      summary = get_naive_rag_summary(article)
      print("NAIVE RAG DONE_____________________________________________________________")
      df.at[current_abstract, 'naïve_rag']=summary
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED------------------------&&&&&&^^^^^^^%%%%%%%%%%%-----------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  while True:
    try:
      llm=ChatGroq(model=model,temperature=0)
      res=llm.invoke(f"Summarize the following {article}").content
      print("ZERO SHOT DONE DONE_____________________________________________________________")
      df.at[current_abstract, 'zero-shot']=res
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED------------------------&&&&&&^^^^^^^%%%%%%%%%%%-----------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  df.to_excel(file_path,index=False)
  current_abstract+=1




In [ ]:
current_key+=1
os.environ["GROQ_API_KEY"]=groq_key[current_key]

In [ ]:
print(len(groq_key),current_key)

In [ ]:
print(groq_key[current_key])

In [ ]:
folder_path = '/content/drive/MyDrive/Graph_Rag_Testing'
file_name='BENCHMARKING.xlsx'
if os.path.exists(folder_path):
  file_path=os.path.join(folder_path,file_name)
  df=pd.read_excel(file_path)

In [ ]:
df.iloc[68,10]

#EVALUATION AUTOMATION

In [ ]:
!pip install scispacy langchain langchain-groq langchain-core nltk bert-score rouge-score transformers sentence-transformers
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

nltk.download('punkt')
nltk.download('punkt_tab')
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
nli = pipeline("text-classification",
               model="facebook/bart-large-mnli",
               device=0)
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)


In [ ]:
import os
groq_key=[]
current_key=0
os.environ["GROQ_API_KEY"]=groq_key[current_key]

In [ ]:
import spacy
try:
    nlp = spacy.load("en_ner_bc5cdr_md")
except OSError:
    print("Model not found. Please install the en_ner_bc5cdr_md model.")

def medical_entity_recall(reference, summary):
    ref_doc = nlp(reference)
    sum_doc = nlp(summary)
    ref_entities = set([ent.text.lower() for ent in ref_doc.ents])
    sum_entities = set([ent.text.lower() for ent in sum_doc.ents])
    if not ref_entities:
        return 0.0
    matched_entities = ref_entities.intersection(sum_entities)

    recall = len(matched_entities) / len(ref_entities)

    return recall
def bleu(reference, summary):

    ref_tokens = [word_tokenize(reference)]
    sum_tokens = word_tokenize(summary)

    return sentence_bleu(ref_tokens, sum_tokens)

def bert_score_metric(reference, summary):

    P, R, F1 = bertscore([summary],[reference], lang="en")

    return F1.item()
def rouge_l(reference, summary):

    scores = rouge.score(reference, summary)

    return scores["rougeL"].fmeasure
def coverage_score(article, summary):
    article_sents = sent_tokenize(article)
    summary_sents = sent_tokenize(summary)
    if len(article_sents) == 0:
        return 0

    article_emb = embedding_model.encode(article_sents, convert_to_tensor=True)
    summary_emb = embedding_model.encode(summary_sents, convert_to_tensor=True)
    sim = util.cos_sim(article_emb, summary_emb)
    matched = (sim.max(dim=1).values > 0.6).sum().item()
    return matched / len(article_sents)



old-one

In [ ]:
import os
import time
import sys
from typing import List, Dict
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from groq import RateLimitError
import nltk

# --- CONFIGURATION & KEY ROTATION SETUP ---


class MedicalAtomizer:
    """
    Decomposes a medical summary into atomic facts using LangChain + Groq.
    Includes rate limit handling and key rotation.
    """
    def __init__(self, model_name="llama-3.3-70b-versatile"):
        self.model_name = model_name
        try:
            nltk.data.find('tokenizers/punkt')
        except LookupError:
            print("Downloading NLTK sentence tokenizer...")
            nltk.download('punkt')
            nltk.download('punkt_tab')

        # 1. DEFINE PROMPT FIRST
        self.few_shot_prompt = """
You are an expert medical linguistic parser. Decompose the given sentence into independent, atomic facts.
Rules:
1. Split compound sentences.
2. Isolate numerical values (dosages, p-values, percentages).
3. Keep proper nouns intact.
4. Output a bulleted list starting with "- ".
**CRITICAL: Treat decimal numbers (e.g., 33.7, 0.05) as single entities. NEVER split a number at the decimal point.**
Preserve the full context for every number (e.g., "33.7% in the US" -> "- CONDITION/PHENOMENON in the US is 33.7%", NOT "- The percentage is 33").

If a sentence lists multiple values for multiple groups (e.g., 'Values were X, Y, and Z for groups A, B, and C respectively'), you MUST explicitly link each value to its group. (e.g., '- The value for Group A was X. - The value for Group B was Y.')

**Ignore generic statements about the existence of people or roles (e.g., 'There was a surgeon').** Only extract statements containing clinical findings, data, or specific procedural details. Ensure means include their standard deviations (e.g., '66 ± 10 mins') if present.
--- EXAMPLES ---

Sentence: "Thierry Henry (born 17 August 1977) is a French professional football coach."
- Thierry Henry was born on 17 August 1977.
- Thierry Henry is a French professional football coach.

Sentence: "Aspirin, taken at 50mg daily, reduced headache frequency by 20% in the study group."
- The dosage of Aspirin was 50mg daily.
- Aspirin reduced headache frequency by 20%.
- The reduction was observed in the study group.

Sentence: "The patient, a 45-year-old male, presented with acute dyspnea and chest pain."
- The patient was a male.
- The patient was 45 years old.
- The patient presented with acute dyspnea.
- The patient presented with chest pain.

Sentence: "Results showed a statistically significant reduction in HbA1c (p<0.05)."
- The results showed a reduction in HbA1c.
- The reduction was statistically significant.
- The p-value was less than 0.05.

Sentence: "Adverse events included nausea (5%), vomiting (3%), and dizziness."
- Adverse events included nausea.
- Nausea occurred in 5% of cases.
- Adverse events included vomiting.
- Vomiting occurred in 3% of cases.
- Adverse events included dizziness.

Sentence: "Drug X is an ACE inhibitor used for hypertension."
- Drug X is an ACE inhibitor.
- Drug X is used for hypertension.

Sentence: "The study was a randomized, double-blind, placebo-controlled trial."
- The study was a randomized trial.
- The study was double-blind.
- The study was placebo-controlled.

Sentence: "Participants were recruited from 5 hospitals in London between 2020 and 2022."
- Participants were recruited from hospitals in London.
- The number of hospitals was 5.
- Recruitment occurred between 2020 and 2022.

Sentence: "Mean systolic blood pressure dropped from 140 mmHg to 125 mmHg."
- Mean systolic blood pressure dropped.
- The baseline pressure was 140 mmHg.
- The final pressure was 125 mmHg.

Sentence: "Metformin is the first-line treatment for T2DM, primarily lowering glucose production."
- Metformin is a treatment for T2DM.
- Metformin is a first-line treatment.
- Metformin lowers glucose production.
- Lowering glucose production is its primary mechanism.

Sentence: "The tumor measured 3x4 cm and was located in the left lung."
- The tumor measured 3x4 cm.
- The tumor was located in the lung.
- The location was the left lung.

Sentence: "No significant difference was found between the two groups (p=0.45)."
- No significant difference was found between the groups.
- The p-value was 0.45.

Sentence: "He has a history of asthma, COPD, and smoking."
- He has a history of asthma.
- He has a history of COPD.
- He has a history of smoking.

Sentence: "The intervention group received 10mg, while the control group received a placebo."
- The intervention group received 10mg.
- The control group received a placebo.

Sentence: "Diagnosis was confirmed via MRI and CT scan."
- Diagnosis was confirmed via MRI.
- Diagnosis was confirmed via CT scan.

Sentence: "Symptoms persisted for 3 weeks despite antibiotic therapy."
- Symptoms persisted for 3 weeks.
- The patient received antibiotic therapy.
- Symptoms persisted despite therapy.

Sentence: "Total cholesterol levels decreased by 15 mg/dL."
- Total cholesterol levels decreased.
- The decrease was 15 mg/dL.

Sentence: "The prevalence of diabetes in the cohort was 12%."
- The prevalence of diabetes was 12%.
- This prevalence was observed in the cohort.

Sentence: "Surgery was performed under general anesthesia."
- Surgery was performed.
- General anesthesia was used.

Sentence: "Post-operative complications included infection and hemorrhage."
- Post-operative complications included infection.
- Post-operative complications included hemorrhage.

Sentence: "The mortality rate was lower in Group A (2%) compared to Group B (8%)."
- The mortality rate in Group A was 2%.
- The mortality rate in Group B was 8%.
- The mortality rate was lower in Group A than Group B.

Sentence: "Patients with renal failure were excluded from the study."
- Patients with renal failure were excluded.

Sentence: "Insulin glargine was administered subcutaneously once daily."
- Insulin glargine was administered.
- Administration was subcutaneous.
- Frequency was once daily.

Sentence: "The median survival time was 18 months (95% CI: 14-22)."
- The median survival time was 18 months.
- The 95% Confidence Interval was 14-22 months.

Sentence: "Common side effects are headache and fatigue."
- A common side effect is headache.
- A common side effect is fatigue.

Sentence: "The virus spreads via respiratory droplets."
- The virus spreads via respiratory droplets.

Sentence: "Treatment adherence was 85%."
- Treatment adherence was measured.
- The adherence rate was 85%.

Sentence: "Genetic testing revealed a BRCA1 mutation."
- Genetic testing was performed.
- The testing revealed a BRCA1 mutation.

Sentence: "The drug was approved by the FDA in 2023."
- The drug was approved by the FDA.
- Approval occurred in 2023.

Sentence: "Obesity prevalence is 33.7% in the US, 32.1% in England, and 7.2% in Taiwan."
- The obesity prevalence in the US is 33.7%.
- The obesity prevalence in England is 32.1%.
- The obesity prevalence in Taiwan is 7.2%.

Sentence: "Underweight percentages are 1.7%, 0.8%, and 3.4% respectively."
- The underweight percentage for the first group is 1.7%.
- The underweight percentage for the second group is 0.8%.
- The underweight percentage for the third group is 3.4%.

Sentence: "Dr. Smith led the research team at Mayo Clinic."
- Dr. Smith led the research team.
- The team was at Mayo Clinic.

--- END EXAMPLES ---

Sentence: "{sentence}"
"""
        # 2. THEN INITIALIZE CHAIN
        self._reinit_chain()

    def _reinit_chain(self):
        """Re-initializes the LLM and chain to pick up the new API Key from os.environ"""
        self.llm = ChatGroq(model=self.model_name, temperature=0.0, max_retries=2)
        self.prompt_template = ChatPromptTemplate.from_template(self.few_shot_prompt)
        self.chain = self.prompt_template | self.llm | StrOutputParser()

    def run(self, summary: str) -> List[str]:
        global current_key
        # Simple sentence splitting
        sentences = nltk.sent_tokenize(summary)

        # Filter out tiny fragments (like bullet points or artifacts)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
        all_facts = []

        print(f"   [Atomizer] Breaking down {len(sentences)} sentences...")

        for sent in sentences:
            while True:
                try:
                    result = self.chain.invoke({"sentence": sent})

                    # Parse output (lines starting with -)
                    facts = [line.strip("- ").strip() for line in result.split("\n") if line.strip().startswith("-")]
                    all_facts.extend(facts)
                    break # Success, exit while loop

                except RateLimitError:
                    if current_key < len(groq_key) - 1:
                        current_key += 1
                        print("KEY CHANGED-----------------------------------------------,current key:", current_key)
                        os.environ["GROQ_API_KEY"] = groq_key[current_key]
                        self._reinit_chain() # Rebuild chain with new key
                    else:
                        print("All keys exhausted. Exiting.")
                        sys.exit() # Or handle as you wish

                except Exception as e:
                    print(f"   [Error] Could not atomize sentence: {sent[:30]}... ({e})")
                    all_facts.append(sent) # Fallback
                    current_key += 1
                    print("KEY CHANGED-----------------------------------------------,current key:", current_key)
                    os.environ["GROQ_API_KEY"] = groq_key[current_key]
                    self._reinit_chain()
                    break # Exit loop on non-rate-limit error

        return all_facts

class MedicalVerifier:
    """
    Verifies facts against the source article using LangChain + Groq.
    Includes rate limit handling and key rotation.
    """
    def __init__(self, model_name="llama-3.3-70b-versatile"):
        self.model_name = model_name

        # 1. DEFINE PROMPT FIRST
        template = """
You are an expert medical fact-checker.
Verify if the [Statement] is explicitly supported by the [Article].

[Article Snippet]
{article}

[Statement]
{atom}

[Instructions]
- Output ONLY "TRUE" if the statement is fully supported by the text.
- Output ONLY "FALSE" if the statement is contradicted OR not mentioned.
- Be strictly literal about numbers (e.g., 50mg != 500mg).

Answer (TRUE/FALSE):
"""
        self.prompt = ChatPromptTemplate.from_template(template)
        self._reinit_chain()

    def _reinit_chain(self):
        """Re-initializes the LLM and chain to pick up the new API Key"""
        self.llm = ChatGroq(model=self.model_name, temperature=0.0, max_retries=2)
        self.chain = self.prompt | self.llm | StrOutputParser()

    def verify(self, atom: str, source_article: str) -> bool:
        global current_key
        truncated_source = source_article

        while True:
            try:
                result = self.chain.invoke({"article": truncated_source, "atom": atom})
                clean_result = result.strip().upper()
                return "TRUE" in clean_result and "FALSE" not in clean_result

            except RateLimitError:
                if current_key < len(groq_key) - 1:
                    current_key += 1
                    print("KEY CHANGED-----------------------------------------------,current key:", current_key)
                    os.environ["GROQ_API_KEY"] = groq_key[current_key]
                    self._reinit_chain() # Rebuild chain with new key
                else:
                    print("All keys exhausted. Exiting.")
                    sys.exit()

            except Exception as e:
                print(f"   [Error] Verification failed for atom: {atom[:20]}... ({e})")
                current_key += 1
                print("KEY CHANGED-----------------------------------------------,current key:", current_key)
                os.environ["GROQ_API_KEY"] = groq_key[current_key]
                self._reinit_chain()
                return False # Default to False on error

class MedicalFactScore:
    def __init__(self):
        self.atomizer = MedicalAtomizer()
        self.verifier = MedicalVerifier()

    def calculate(self, summary: str, source_article: str) -> Dict:
        print("1. Atomizing Summary...")
        atomic_facts = self.atomizer.run(summary)
        print(f"   -> Generated {len(atomic_facts)} atomic facts.")

        print("2. Verifying Facts...")
        results = []
        supported_count = 0

        for fact in atomic_facts:
            is_supported = self.verifier.verify(fact, source_article)
            status = "VERIFIED:" if is_supported else "NOT VERIFIED:"
            print(f"   {status} {fact}")

            results.append({
                "fact": fact,
                "is_supported": is_supported
            })
            if is_supported:
                supported_count += 1

        score = supported_count / len(atomic_facts) if atomic_facts else 0.0
        return {"score": score, "details": results}

# if __name__ == "__main__":
#     article_input = """
#     """

#     summary_output = """
#     """


#     pipeline = MedicalFactScore()

#     start = time.time()
#     report = pipeline.calculate(summary_output, article_input)

#     print("\n" + "="*40)
#     print(f"FINAL FACTSCORE: {report['score']:.2%}")
#     print("="*40)
#     print(f"Time: {time.time() - start:.2f}s")

mini-batching


In [ ]:
import os
import sys
import time
import json
from typing import List, Dict
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from groq import RateLimitError
import nltk

class MedicalAtomizer:
    """
    Decomposes a medical summary into atomic facts using LangChain + Groq.
    Includes rate limit handling and key rotation.
    """
    def __init__(self, model_name="llama-3.3-70b-versatile"):
        self.model_name = model_name
        try:
            nltk.data.find('tokenizers/punkt')
        except LookupError:
            print("Downloading NLTK sentence tokenizer...")
            nltk.download('punkt')
            nltk.download('punkt_tab')

        # 1. DEFINE PROMPT FIRST
        self.few_shot_prompt = """
You are an expert medical linguistic parser. Decompose the given sentence into independent, atomic facts.
Rules:
1. Split compound sentences.
2. Isolate numerical values (dosages, p-values, percentages).
3. Keep proper nouns intact.
4. Output a bulleted list starting with "- ".
**CRITICAL: Treat decimal numbers (e.g., 33.7, 0.05) as single entities. NEVER split a number at the decimal point.**
Preserve the full context for every number (e.g., "33.7% in the US" -> "- CONDITION/PHENOMENON in the US is 33.7%", NOT "- The percentage is 33").

If a sentence lists multiple values for multiple groups (e.g., 'Values were X, Y, and Z for groups A, B, and C respectively'), you MUST explicitly link each value to its group. (e.g., '- The value for Group A was X. - The value for Group B was Y.')

**Ignore generic statements about the existence of people or roles (e.g., 'There was a surgeon').** Only extract statements containing clinical findings, data, or specific procedural details. Ensure means include their standard deviations (e.g., '66 ± 10 mins') if present.
--- EXAMPLES ---

Sentence: "Thierry Henry (born 17 August 1977) is a French professional football coach."
- Thierry Henry was born on 17 August 1977.
- Thierry Henry is a French professional football coach.

Sentence: "Aspirin, taken at 50mg daily, reduced headache frequency by 20% in the study group."
- The dosage of Aspirin was 50mg daily.
- Aspirin reduced headache frequency by 20%.
- The reduction was observed in the study group.

Sentence: "The patient, a 45-year-old male, presented with acute dyspnea and chest pain."
- The patient was a male.
- The patient was 45 years old.
- The patient presented with acute dyspnea.
- The patient presented with chest pain.

Sentence: "Results showed a statistically significant reduction in HbA1c (p<0.05)."
- The results showed a reduction in HbA1c.
- The reduction was statistically significant.
- The p-value was less than 0.05.

Sentence: "Adverse events included nausea (5%), vomiting (3%), and dizziness."
- Adverse events included nausea.
- Nausea occurred in 5% of cases.
- Adverse events included vomiting.
- Vomiting occurred in 3% of cases.
- Adverse events included dizziness.

Sentence: "Drug X is an ACE inhibitor used for hypertension."
- Drug X is an ACE inhibitor.
- Drug X is used for hypertension.

Sentence: "The study was a randomized, double-blind, placebo-controlled trial."
- The study was a randomized trial.
- The study was double-blind.
- The study was placebo-controlled.

Sentence: "Participants were recruited from 5 hospitals in London between 2020 and 2022."
- Participants were recruited from hospitals in London.
- The number of hospitals was 5.
- Recruitment occurred between 2020 and 2022.

Sentence: "Mean systolic blood pressure dropped from 140 mmHg to 125 mmHg."
- Mean systolic blood pressure dropped.
- The baseline pressure was 140 mmHg.
- The final pressure was 125 mmHg.

Sentence: "Metformin is the first-line treatment for T2DM, primarily lowering glucose production."
- Metformin is a treatment for T2DM.
- Metformin is a first-line treatment.
- Metformin lowers glucose production.
- Lowering glucose production is its primary mechanism.

Sentence: "The tumor measured 3x4 cm and was located in the left lung."
- The tumor measured 3x4 cm.
- The tumor was located in the lung.
- The location was the left lung.

Sentence: "No significant difference was found between the two groups (p=0.45)."
- No significant difference was found between the groups.
- The p-value was 0.45.

Sentence: "He has a history of asthma, COPD, and smoking."
- He has a history of asthma.
- He has a history of COPD.
- He has a history of smoking.

Sentence: "The intervention group received 10mg, while the control group received a placebo."
- The intervention group received 10mg.
- The control group received a placebo.

Sentence: "Diagnosis was confirmed via MRI and CT scan."
- Diagnosis was confirmed via MRI.
- Diagnosis was confirmed via CT scan.

Sentence: "Symptoms persisted for 3 weeks despite antibiotic therapy."
- Symptoms persisted for 3 weeks.
- The patient received antibiotic therapy.
- Symptoms persisted despite therapy.

Sentence: "Total cholesterol levels decreased by 15 mg/dL."
- Total cholesterol levels decreased.
- The decrease was 15 mg/dL.

Sentence: "The prevalence of diabetes in the cohort was 12%."
- The prevalence of diabetes was 12%.
- This prevalence was observed in the cohort.

Sentence: "Surgery was performed under general anesthesia."
- Surgery was performed.
- General anesthesia was used.

Sentence: "Post-operative complications included infection and hemorrhage."
- Post-operative complications included infection.
- Post-operative complications included hemorrhage.

Sentence: "The mortality rate was lower in Group A (2%) compared to Group B (8%)."
- The mortality rate in Group A was 2%.
- The mortality rate in Group B was 8%.
- The mortality rate was lower in Group A than Group B.

Sentence: "Patients with renal failure were excluded from the study."
- Patients with renal failure were excluded.

Sentence: "Insulin glargine was administered subcutaneously once daily."
- Insulin glargine was administered.
- Administration was subcutaneous.
- Frequency was once daily.

Sentence: "The median survival time was 18 months (95% CI: 14-22)."
- The median survival time was 18 months.
- The 95% Confidence Interval was 14-22 months.

Sentence: "Common side effects are headache and fatigue."
- A common side effect is headache.
- A common side effect is fatigue.

Sentence: "The virus spreads via respiratory droplets."
- The virus spreads via respiratory droplets.

Sentence: "Treatment adherence was 85%."
- Treatment adherence was measured.
- The adherence rate was 85%.

Sentence: "Genetic testing revealed a BRCA1 mutation."
- Genetic testing was performed.
- The testing revealed a BRCA1 mutation.

Sentence: "The drug was approved by the FDA in 2023."
- The drug was approved by the FDA.
- Approval occurred in 2023.

Sentence: "Obesity prevalence is 33.7% in the US, 32.1% in England, and 7.2% in Taiwan."
- The obesity prevalence in the US is 33.7%.
- The obesity prevalence in England is 32.1%.
- The obesity prevalence in Taiwan is 7.2%.

Sentence: "Underweight percentages are 1.7%, 0.8%, and 3.4% respectively."
- The underweight percentage for the first group is 1.7%.
- The underweight percentage for the second group is 0.8%.
- The underweight percentage for the third group is 3.4%.

Sentence: "Dr. Smith led the research team at Mayo Clinic."
- Dr. Smith led the research team.
- The team was at Mayo Clinic.

--- END EXAMPLES ---

Sentence: "{sentence}"
"""
        # 2. THEN INITIALIZE CHAIN
        self._reinit_chain()

    def _reinit_chain(self):
        """Re-initializes the LLM and chain to pick up the new API Key from os.environ"""
        self.llm = ChatGroq(model=self.model_name, temperature=0.0, max_retries=2)
        self.prompt_template = ChatPromptTemplate.from_template(self.few_shot_prompt)
        self.chain = self.prompt_template | self.llm | StrOutputParser()

    def run(self, summary: str) -> List[str]:
        global current_key
        # Simple sentence splitting
        sentences = nltk.sent_tokenize(summary)

        # Filter out tiny fragments (like bullet points or artifacts)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
        all_facts = []

        print(f"   [Atomizer] Breaking down {len(sentences)} sentences...")

        for sent in sentences:
            while True:
                try:
                    result = self.chain.invoke({"sentence": sent})

                    # Parse output (lines starting with -)
                    facts = [line.strip("- ").strip() for line in result.split("\n") if line.strip().startswith("-")]
                    all_facts.extend(facts)
                    break # Success, exit while loop

                except RateLimitError:
                    if current_key < len(groq_key) - 1:
                        current_key += 1
                        print("KEY CHANGED-----------------------------------------------,current key:", current_key)
                        os.environ["GROQ_API_KEY"] = groq_key[current_key]
                        self._reinit_chain() # Rebuild chain with new key
                    else:
                        print("All keys exhausted. Exiting.")
                        sys.exit() # Or handle as you wish

                except Exception as e:
                    print(f"   [Error] Could not atomize sentence: {sent[:30]}... ({e})")
                    all_facts.append(sent) # Fallback
                    current_key += 1
                    print("KEY CHANGED-----------------------------------------------,current key:", current_key)
                    os.environ["GROQ_API_KEY"] = groq_key[current_key]
                    self._reinit_chain()
                    break # Exit loop on non-rate-limit error

        return all_facts


class MedicalVerifier:
    def __init__(self, model_name="llama-3.3-70b-versatile"):
        self.model_name = model_name
        self._reinit_chain()

    def _reinit_chain(self):
        template = """
You are an expert medical fact-checker.
Verify if each statement in the list is explicitly supported by the [Article].

[Article Snippet]
{article}

[Statements to Verify]
{atoms}

[Instructions]
- Evaluate EACH statement independently against the article.
- Be strictly literal about clinical findings, schemas, and numbers.
- Output ONLY a valid JSON object.
- The JSON object must contain a single key "verifications", which is a list of objects.
- Each object must have exactly two keys: "fact" (string, the exact text of the statement) and "is_supported" (boolean).

JSON Output:
"""
        self.prompt = ChatPromptTemplate.from_template(template)
        self.llm = ChatGroq(
            model=self.model_name,
            temperature=0.0,
            max_retries=2,
            model_kwargs={"response_format": {"type": "json_object"}}
        )
        self.chain = self.prompt | self.llm | StrOutputParser()

    def verify_single(self, fact: str, source_article: str) -> Dict:
        """Fallback method for 1-by-1 verification if a batch fails completely."""
        global current_key
        # We wrap the single fact in a list so the prompt handles it identically
        formatted_fact = f"1. {fact}"

        while True:
            try:
                result = self.chain.invoke({"article": source_article, "atoms": formatted_fact})
                parsed_json = json.loads(result)
                verifications = parsed_json.get("verifications", [])

                if len(verifications) == 1:
                    return verifications[0]
                else:
                    return {"fact": fact, "is_supported": False} # Failsafe

            except RateLimitError:
                if current_key < len(groq_key) - 1:
                    current_key += 1
                    os.environ["GROQ_API_KEY"] = groq_key[current_key]
                    self._reinit_chain()
                else:
                    sys.exit("All keys exhausted.")
            except Exception:
                # If even a single prompt fails after retries, default to False
                return {"fact": fact, "is_supported": False}

    def verify_batch(self, facts_batch: List[str], source_article: str) -> List[Dict]:
        global current_key
        retry_count = 0
        max_retries = 3

        # NO TRUNCATION. We pass the full source_article every time.
        formatted_facts = "\n".join([f"{i+1}. {fact}" for i, fact in enumerate(facts_batch)])

        while retry_count <= max_retries:
            try:
                result = self.chain.invoke({"article": source_article, "atoms": formatted_facts})
                parsed_json = json.loads(result)
                batch_results = parsed_json.get("verifications", [])

                # Strict length validation
                if len(batch_results) != len(facts_batch):
                    print(f"   [Warning] Model dropped facts. Expected {len(facts_batch)}, got {len(batch_results)}. Retrying (Attempt {retry_count + 1}/{max_retries})...")
                    retry_count += 1
                    continue
                #added this check it
                # The zip mapping fix to guarantee zero string drift
                mapped_results = []
                for original_fact, llm_result in zip(facts_batch, batch_results):
                    mapped_results.append({
                        "fact": original_fact,
                        "is_supported": llm_result.get("is_supported", False)
                    })
                return mapped_results

                #return batch_results

            except RateLimitError:
                if current_key < len(groq_key) - 1:
                    current_key += 1
                    print("KEY CHANGED-----------------------------------------------,current key:", current_key)
                    os.environ["GROQ_API_KEY"] = groq_key[current_key]
                    self._reinit_chain()
                else:
                    print("All keys exhausted. Exiting.")
                    sys.exit()

            except (json.JSONDecodeError, Exception) as e:
                print(f"   [Error] Batch failed: {e}. Retrying (Attempt {retry_count + 1}/{max_retries})...")
                retry_count += 1
                continue

        print(f"   [Fallback] Batch failed {max_retries} times. Falling back to sequential 1-by-1 verification for these {len(facts_batch)} facts to preserve accuracy.")
        fallback_results = []
        for fact in facts_batch:
            fallback_results.append(self.verify_single(fact, source_article))
            time.sleep(0.5)

        return fallback_results
class MedicalFactScore:
    """Manages the Atomizer and Verifier to generate the final FActScore."""
    def __init__(self, batch_size=15):
        self.atomizer = MedicalAtomizer()
        self.verifier = MedicalVerifier()
        self.batch_size = batch_size

    def calculate(self, summary: str, source_article: str) -> Dict:
        print("1. Atomizing Summary...")
        atomic_facts = self.atomizer.run(summary)
        print(f"   -> Generated {len(atomic_facts)} atomic facts.")

        print(f"2. Verifying Facts in chunks of {self.batch_size}...")
        results = []
        supported_count = 0

        for i in range(0, len(atomic_facts), self.batch_size):
            batch = atomic_facts[i:i + self.batch_size]
            print(f"   [Batch] Processing facts {i+1} to {min(i+self.batch_size, len(atomic_facts))}...")

            batch_results = self.verifier.verify_batch(batch, source_article)

            for item in batch_results:
                is_supported = item.get("is_supported", False)
                fact_text = item.get("fact", "Unknown Fact")

                status = "VERIFIED:" if is_supported else "NOT VERIFIED:"
                print(f"      {status} {fact_text}")

                results.append(item)
                if is_supported:
                    supported_count += 1

        score = supported_count / len(atomic_facts) if atomic_facts else 0.0
        return {"score": score, "details": results}


# if __name__=="__main__":
#   pipeline = MedicalFactScore(batch_size=15)

#     start = time.time()
#     report = pipeline.calculate(summary_output, article_input)

#     print("\n" + "="*40)
#     print(f"FINAL FACTSCORE: {report['score']:.2%}")
#     print("="*40)
#     print(f"Execution Time: {time.time() - start:.2f}s")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import time
import sys
import pandas as pd


INPUT_FILE = '/content/drive/MyDrive/Graph_Rag_Testing/BENCHMARKING.xlsx'
OUTPUT_FILE = '/content/drive/MyDrive/Graph_Rag_Testing/benchmarking_results_final.xlsx'


SYSTEMS_TO_EVALUATE = ['zero-shot', 'naïve_rag', 'naïve_graphrag', 'SYSTEM']

def run_colab_evaluation_pipeline(start_index: int = 0):
    """
    Evaluates 4 summarization systems across multiple metrics and auto-saves progress.
    """
    # Load dataset: Resume from output if it exists, otherwise start fresh from input
    if os.path.exists(OUTPUT_FILE):
        print(f"Resuming from existing output file: {OUTPUT_FILE}")
        df = pd.read_excel(OUTPUT_FILE)
    elif os.path.exists(INPUT_FILE):
        print(f"Starting fresh evaluation from: {INPUT_FILE}")
        df = pd.read_excel(INPUT_FILE)
    else:
        print(f"Error: Could not find {INPUT_FILE}. Please check your Drive path.")
        return

    # Initialize the LLM FactScore pipeline once outside the loop
    pipeline = MedicalFactScore(batch_size=15)
    current_row = start_index

    while current_row < len(df):
        print(f"\n{'='*60}")
        print(f"Processing row {current_row + 1} of {len(df)}...")
        print(f"{'='*60}")
        article_input = str(df.loc[current_row, 'article'])
        for sys_col in SYSTEMS_TO_EVALUATE:
            print(f"\n--- Evaluating System: [{sys_col}] ---")

            summary_output = str(df.loc[current_row, sys_col])
            if not summary_output.strip() or summary_output.lower() == 'nan':
                print(f"Skipping {sys_col} (Empty summary).")
                continue

            start_time = time.time()


            report = pipeline.calculate(summary_output, article_input)


            med_recall = medical_entity_recall(article_input, summary_output)
            b_score = bleu(article_input, summary_output)
            bert_s = bert_score_metric(article_input, summary_output)
            r_l = rouge_l(article_input, summary_output)
            cov_score = coverage_score(article_input, summary_output)

            print(f"  > FACTSCORE:       {report['score']:.2%}")
            print(f"  > Med Entity Rec:  {med_recall:.2%}")
            print(f"  > BERTScore:       {bert_s:.4f}")
            print(f"  > ROUGE-L:         {r_l:.4f}")
            print(f"  > Time Taken:      {time.time() - start_time:.2f}s")

            df.loc[current_row, f"{sys_col}_FactScore"] = report['score']
            df.loc[current_row, f"{sys_col}_MedRecall"] = med_recall
            df.loc[current_row, f"{sys_col}_BLEU"] = b_score
            df.loc[current_row, f"{sys_col}_BERTScore"] = bert_s
            df.loc[current_row, f"{sys_col}_ROUGE_L"] = r_l
            df.loc[current_row, f"{sys_col}_Coverage"] = cov_score


        df.to_excel(OUTPUT_FILE, index=False)
        print(f"\n[+] Row {current_row + 1} complete. Progress saved to Drive.")

        current_row += 1

    print("\nBenchmarking complete. All 4 systems evaluated.")



In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
#nlp = spacy.load("en_core_web_sm")
run_colab_evaluation_pipeline(start_index=59)

In [ ]:
df=pd.read_excel('/content/drive/MyDrive/Graph_Rag_Testing/benchmarking_results_final.xlsx')

In [ ]:
df.iloc[58,26]

# VISUALIZATION

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import seaborn as sns


df.columns = df.columns.str.strip()

metrics = ['zero-shot_MedRecall', 'SYSTEM_MedRecall', 'numeric_count']
for col in metrics:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df_clean = df.dropna(subset=['zero-shot_MedRecall', 'SYSTEM_MedRecall', 'numeric_count', 'length_bucket']).copy()

print(f"Data cleaned. Analyzing {len(df_clean)} valid rows...\n")
print("-" * 50)



print("=== 1. Paired T-Test (MedRecall) ===")
t_stat, p_val = stats.ttest_rel(df_clean['zero-shot_MedRecall'], df_clean['SYSTEM_MedRecall'])
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val:.4e}\n")
print("=== 2. Correlation Coefficients (r) ===")
r_zero, p_zero = stats.pearsonr(df_clean['numeric_count'], df_clean['zero-shot_MedRecall'])
r_sys, p_sys = stats.pearsonr(df_clean['numeric_count'], df_clean['SYSTEM_MedRecall'])
print(f"Zero-Shot vs Numeric Count: r = {r_zero:.4f}, p-value = {p_zero:.4f}")
print(f"SYSTEM vs Numeric Count:    r = {r_sys:.4f}, p-value = {p_sys:.4f}\n")

print("=== 3. Effect Size (Cohen's d) ===")
diff = df_clean['SYSTEM_MedRecall'] - df_clean['zero-shot_MedRecall']
cohens_d = diff.mean() / diff.std()
print(f"Cohen's d: {cohens_d:.4f}\n")

print("=== 4. Two-Way ANOVA ===")
df_melt = df_clean.melt(id_vars=['length_bucket'],
                        value_vars=['zero-shot_MedRecall', 'SYSTEM_MedRecall'],
                        var_name='Model',
                        value_name='MedRecall')

df_melt['Model'] = df_melt['Model'].map({'zero-shot_MedRecall': 'Zero_Shot', 'SYSTEM_MedRecall': 'SYSTEM'})

model = ols('MedRecall ~ C(length_bucket) + C(Model) + C(length_bucket):C(Model)', data=df_melt).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)
print("-" * 50)
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.regplot(data=df_clean, x='numeric_count', y='zero-shot_MedRecall', ax=axes[0],
            label='Zero-Shot', color='#d62728', scatter_kws={'alpha':0.5})
sns.regplot(data=df_clean, x='numeric_count', y='SYSTEM_MedRecall', ax=axes[0],
            label='SYSTEM', color='#1f77b4', scatter_kws={'alpha':0.5})

axes[0].set_title('Robustness to Quantitative Density', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Numeric Count in Text', fontsize=12)
axes[0].set_ylabel('MedRecall Score', fontsize=12)
axes[0].legend()
order = ['short', 'medium', 'long', 'very_long']
existing_order = [o for o in order if o in df_clean['length_bucket'].unique()]

sns.pointplot(data=df_melt, x='length_bucket', y='MedRecall', hue='Model', ax=axes[1],
              order=existing_order, palette={'Zero_Shot': '#d62728', 'SYSTEM': '#1f77b4'})

axes[1].set_title('Interaction Plot: Performance Scalability by Length', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Document Length', fontsize=12)
axes[1].set_ylabel('MedRecall Score', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:


import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import seaborn as sns
import re

from google.colab import files
uploaded = files.upload()
FILE_PATH = list(uploaded.keys())[0]
SHEET_NAME = "RESULT"

METRICS = ["FactScore", "MedRecall", "BLEU", "BERTScore", "ROUGE_L", "Coverage"]
TARGET = "SYSTEM"


df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df.columns = df.columns.str.strip()

metric_cols = [c for c in df.columns if any(c.endswith(f"_{m}") for m in METRICS)]
prefixes = []
for c in metric_cols:
    for m in METRICS:
        if c.endswith(f"_{m}"):
            prefix = c[: -(len(m) + 1)]
            if prefix not in prefixes:
                prefixes.append(prefix)
            break

BASELINES = [p for p in prefixes if p != TARGET]
print("Detected baseline structures:", BASELINES)
print("Detected target system:", TARGET)
for col in metric_cols + ["numeric_count", "word_count"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
required_cols = ["length_bucket", "numeric_count"] + metric_cols
df_clean = df.dropna(subset=required_cols).copy()

print(f"\nData cleaned. {len(df_clean)} valid rows out of {len(df)} total rows.\n")
print("-" * 70)


def sig_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return "ns"



ttest_rows = []
for baseline in BASELINES:
    for metric in METRICS:
        col_b = f"{baseline}_{metric}"
        col_s = f"{TARGET}_{metric}"
        if col_b not in df_clean.columns or col_s not in df_clean.columns:
            continue

        a = df_clean[col_b]
        b = df_clean[col_s]
        n = len(df_clean)

        t_stat, p_val = stats.ttest_rel(a, b)

        diff = b - a  # SYSTEM - baseline
        cohens_d = diff.mean() / diff.std(ddof=1)

        ttest_rows.append({
            "Comparison": f"{baseline} vs {TARGET}",
            "Metric": metric,
            "N": n,
            "Mean_Baseline": round(a.mean(), 4),
            "SD_Baseline": round(a.std(ddof=1), 4),
            "Mean_SYSTEM": round(b.mean(), 4),
            "SD_SYSTEM": round(b.std(ddof=1), 4),
            "Mean_Diff(SYS-Base)": round(diff.mean(), 4),
            "t_stat": round(t_stat, 4),
            "df": n - 1,
            "p_value": p_val,
            "Cohens_d": round(cohens_d, 4),
            "Sig": sig_stars(p_val),
        })

ttest_table = pd.DataFrame(ttest_rows)
print("\n=== TABLE 1: Paired t-test & Cohen's d (Baseline vs SYSTEM) ===\n")
print(ttest_table.to_string(index=False))



ALL_STRUCTURES = BASELINES + [TARGET]
corr_rows = []
for metric in METRICS:
    row = {"Metric": metric}
    for structure in ALL_STRUCTURES:
        col = f"{structure}_{metric}"
        if col not in df_clean.columns:
            continue
        r, p = stats.pearsonr(df_clean["numeric_count"], df_clean[col])
        row[f"{structure}_r"] = round(r, 4)
        row[f"{structure}_p"] = round(p, 4)
        row[f"{structure}_sig"] = sig_stars(p)
    corr_rows.append(row)

corr_table = pd.DataFrame(corr_rows)
print("\n\n=== TABLE 2: Pearson correlation — numeric_count vs metric, by structure ===\n")
print(corr_table.to_string(index=False))



def safe_name(s):
    """statsmodels formulas choke on hyphens/accents — give each level a clean alias."""
    return re.sub(r"[^0-9a-zA-Z_]", "_", s)


anova_rows = []
for baseline in BASELINES:
    for metric in METRICS:
        col_b = f"{baseline}_{metric}"
        col_s = f"{TARGET}_{metric}"
        if col_b not in df_clean.columns or col_s not in df_clean.columns:
            continue

        df_melt = df_clean.melt(
            id_vars=["length_bucket"],
            value_vars=[col_b, col_s],
            var_name="Model",
            value_name="Score",
        )
        df_melt["Model"] = df_melt["Model"].map({col_b: safe_name(baseline), col_s: TARGET})
        df_melt["length_bucket"] = df_melt["length_bucket"].map(safe_name)

        model = ols("Score ~ C(length_bucket) + C(Model) + C(length_bucket):C(Model)", data=df_melt).fit()
        at = sm.stats.anova_lm(model, typ=2)

        def grab(effect):
            if effect in at.index:
                return at.loc[effect, "F"], at.loc[effect, "PR(>F)"]
            return np.nan, np.nan

        f_len, p_len = grab("C(length_bucket)")
        f_mod, p_mod = grab("C(Model)")
        f_int, p_int = grab("C(length_bucket):C(Model)")

        anova_rows.append({
            "Comparison": f"{baseline} vs {TARGET}",
            "Metric": metric,
            "length_bucket_F": round(f_len, 4), "length_bucket_p": round(p_len, 4), "length_bucket_sig": sig_stars(p_len),
            "Model_F": round(f_mod, 4), "Model_p": round(p_mod, 4), "Model_sig": sig_stars(p_mod),
            "Interaction_F": round(f_int, 4), "Interaction_p": round(p_int, 4), "Interaction_sig": sig_stars(p_int),
        })

anova_table = pd.DataFrame(anova_rows)
print("\n\n=== TABLE 3: Two-way ANOVA (Model x length_bucket -> Metric) ===\n")
print(anova_table.to_string(index=False))


with pd.ExcelWriter("statistical_analysis_results.xlsx", engine="openpyxl") as writer:
    ttest_table.to_excel(writer, sheet_name="TTest_CohensD", index=False)
    corr_table.to_excel(writer, sheet_name="Pearson_Correlation", index=False)
    anova_table.to_excel(writer, sheet_name="TwoWay_ANOVA", index=False)

ttest_table.to_csv("table1_ttest_cohensd.csv", index=False)
corr_table.to_csv("table2_pearson_correlation.csv", index=False)
anova_table.to_csv("table3_twoway_anova.csv", index=False)

print("\n\nSaved: statistical_analysis_results.xlsx (3 sheets) + 3 standalone CSVs")
print("-" * 70)

plt.style.use("seaborn-v0_8-whitegrid")
pivot_d = ttest_table.pivot(index="Metric", columns="Comparison", values="Cohens_d")
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_d, annot=True, fmt=".2f", cmap="RdBu_r", center=0, cbar_kws={"label": "Cohen's d (SYSTEM - Baseline)"})
plt.title("Effect Size (Cohen's d) — SYSTEM vs each Baseline", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig1_cohens_d_heatmap.png", dpi=150)
plt.show()
METRIC_TO_PLOT = "FactScore"
fig, ax = plt.subplots(figsize=(8, 6))
colors = sns.color_palette("tab10", len(ALL_STRUCTURES))
for structure, color in zip(ALL_STRUCTURES, colors):
    col = f"{structure}_{METRIC_TO_PLOT}"
    if col in df_clean.columns:
        sns.regplot(data=df_clean, x="numeric_count", y=col, ax=ax,
                    label=structure, color=color, scatter_kws={"alpha": 0.4})
ax.set_title(f"Robustness to Numeric Density — {METRIC_TO_PLOT}", fontsize=13, fontweight="bold")
ax.set_xlabel("Numeric Count in Text")
ax.set_ylabel(METRIC_TO_PLOT)
ax.legend()
plt.tight_layout()
plt.savefig("fig2_numeric_density_robustness.png", dpi=150)
plt.show()

order = ["short", "medium", "long", "very_long"]
existing_order = [o for o in order if o in df_clean["length_bucket"].unique()]

fig, axes = plt.subplots(1, len(BASELINES), figsize=(6 * len(BASELINES), 5), sharey=True)
if len(BASELINES) == 1:
    axes = [axes]
for ax, baseline in zip(axes, BASELINES):
    col_b = f"{baseline}_{METRIC_TO_PLOT}"
    col_s = f"{TARGET}_{METRIC_TO_PLOT}"
    df_melt = df_clean.melt(id_vars=["length_bucket"], value_vars=[col_b, col_s],
                             var_name="Model", value_name=METRIC_TO_PLOT)
    df_melt["Model"] = df_melt["Model"].map({col_b: baseline, col_s: TARGET})
    sns.pointplot(data=df_melt, x="length_bucket", y=METRIC_TO_PLOT, hue="Model", ax=ax,
                  order=existing_order)
    ax.set_title(f"{baseline} vs {TARGET}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Document Length")
plt.suptitle(f"Interaction Plot: {METRIC_TO_PLOT} by Length Bucket", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig3_interaction_plots.png", dpi=150)
plt.show()

print("\nDone. Tables: TABLE 1 (t-test/Cohen's d), TABLE 2 (Pearson r), TABLE 3 (two-way ANOVA).")
print("Figures saved as PNGs; full results saved in statistical_analysis_results.xlsx")